# Exploration du jeu de données : *Escherichia coli* - Ciprofloxacine

**Auteur :** Benjamin HIN  
**Projet :** Machine Learning / Deep Learning pour la prédiction de la résistance aux antibiotiques (AMR)  
**Jeu de données :** 

---

### Objectifs du notebook :
1. **Importer** le jeu de données pré-traité .
2. **Afficher** les données et inspecter leur organisation.
3. **Fournir un résumé complet** : dimensions, métadonnées, valeurs manquantes, types de données.
4. **Analyser la variable cible ()** : distribution de la résistance vs sensibilité à la ciprofloxacine.
5. **Explorer les métadonnées génomiques & MLST** (, , ).
6. **Analyser les variables génomiques** (matrice binaire de présence/absence de gènes et clusters).


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
import torch 
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier



# on configure la graine
SEED = 123


# Configuration de l'affichage
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)
sns.set_theme(style="whitegrid", font_scale=1.1)

print("Bibliothèques chargées avec succès.")


Bibliothèques chargées avec succès.


## 1. Importation du jeu de données

### 1.1 Importation des données complète

In [3]:
# Définition du chemin relatif (compatible selon le répertoire courant d'exécution)
data_path = Path("../data/processed/ecoli_ciprofloxacin_snp.parquet")
if not data_path.exists():
    data_path = Path("data/processed/ecoli_ciprofloxacin_snp.parquet")

print(f"Lecture du fichier depuis : {data_path.resolve()}")
df = pd.read_parquet(data_path)
print(f"Jeu de données chargé : {df.shape[0]:,} lignes et {df.shape[1]:,} colonnes.")


Lecture du fichier depuis : /Users/hin.benjamin/Projet_ML_DL/Projet_ML_DL/data/processed/ecoli_ciprofloxacin_snp.parquet
Jeu de données chargé : 5,935 lignes et 20,933 colonnes.


### 1.2 Importation des données splits

In [4]:
# On écrit le chemin du jeu de données (compatible selon le répertoire d'exécution)
DATA = Path("../data/splits_snp_3way")
if not DATA.exists():
    DATA = Path("data/splits_snp_3way")

# On importe les donnnées splits
X_train = pd.read_parquet(DATA/ "X_train.parquet")
y_train = pd.read_parquet(DATA/ "y_train.parquet")
meta_train = pd.read_parquet(DATA/ "metadata_train.parquet")

X_val = pd.read_parquet(DATA/ "X_val.parquet")
y_val = pd.read_parquet(DATA/ "y_val.parquet")["y"]
meta_val = pd.read_parquet(DATA/ "metadata_val.parquet")

X_test = pd.read_parquet(DATA/ "X_test.parquet")
y_test = pd.read_parquet(DATA/ "y_test.parquet")
meta_test = pd.read_parquet(DATA/ "metadata_test.parquet")


# on affiche les tailles des données
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)



(4074, 20929)
(932, 20929)
(929, 20929)


## 2. Machine learning 

### 2.1 Régression logistique 

In [5]:
# on construit le modèle 
model = LogisticRegression(
    random_state = 42, 
    max_iter = 2000
)

model.fit(X_train, y_train)

# On fait les prédictions sur le modèle 
proba_val = model.predict_proba(X_val)[:,1]

# on évalue le modèle sur son pourcentage de bonne association 
y_pred_val = (proba_val >= 0.5)
accuracy = (y_val.to_numpy() == y_pred_val).mean()

# on affiche l'accuracy 
print(f"accuracy de validation = {accuracy}")

# on affiche la matrice de confusion
print(pd.crosstab(y_val, y_pred_val, rownames=["Observé"], colnames=["Prédit"]))

/Users/hin.benjamin/.pyenv/versions/3.12.4/lib/python3.12/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


accuracy de validation = 0.8454935622317596
Prédit   False  True 
Observé              
0          739     37
1          107     49


### 2.2 RandomForrest

In [6]:
# on construit le modèle 
model = RandomForestClassifier(
    random_state = 42
)

model.fit(X_train, y_train)

# On fait les prédictions sur le modèle 
proba_val = model.predict_proba(X_val)[:,1]

# on évalue le modèle sur son pourcentage de bonne association 
y_pred_val = (proba_val >= 0.5)
accuracy = (y_val.to_numpy() == y_pred_val).mean()

# on affiche l'accuracy 
print(f"accuracy de validation = {accuracy}")

# on affiche la matrice de confusion
print(pd.crosstab(y_val, y_pred_val, rownames=["Observé"], colnames=["Prédit"]))

/Users/hin.benjamin/.pyenv/versions/3.12.4/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


accuracy de validation = 0.8969957081545065
Prédit   False  True 
Observé              
0          776      0
1           96     60


### 2.3 XGBoost

In [7]:
# on construit le modèle 
model = XGBClassifier(random_state = 42)
model.fit(X_train, y_train)

# On fait les prédictions sur le modèle 
proba_val = model.predict_proba(X_val)[:,1]

# on évalue le modèle sur son pourcentage de bonne association 
y_pred_val = (proba_val >= 0.5)
accuracy = (y_val.to_numpy() == y_pred_val).mean()

# on affiche l'accuracy 
print(f"accuracy de validation = {accuracy}")

# on affiche la matrice de confusion
print(pd.crosstab(y_val, y_pred_val, rownames=["Observé"], colnames=["Prédit"]))

accuracy de validation = 0.9742489270386266
Prédit   False  True 
Observé              
0          765     11
1           13    143


## 3. Deep learning

### 3.1 Machine learning 